In [1]:

# Install all required libraries
!pip -q install -U yt-dlp
!pip -q install -U openai-whisper
!pip -q install -U google-generativeai
!pip -q install -U pandas
!pip -q install -U ffmpeg-python

# Install FFmpeg
!apt-get -qq update
!apt-get -qq install ffmpeg

print("✅ Libraries Installed Successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 74.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
W: Skipping acquire of configu

In [2]:
import os
import json
import subprocess

import yt_dlp
import whisper
import pandas as pd

print("✅ Libraries Imported Successfully")

✅ Libraries Imported Successfully


In [3]:
folders = [
    "downloads",
    "audio",
    "transcripts",
    "highlights"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created!")

print("\nFolder Structure:")
for folder in folders:
    print("📁", folder)

✅ Project folders created!

Folder Structure:
📁 downloads
📁 audio
📁 transcripts
📁 highlights


In [4]:
youtube_url = input("🎥 Enter YouTube URL: ").strip()

print("\nYou entered:")
print(youtube_url)

🎥 Enter YouTube URL: https://www.youtube.com/watch?v=eVFzbxmKNUw

You entered:
https://www.youtube.com/watch?v=eVFzbxmKNUw


In [7]:
import yt_dlp
import os

def download_video(youtube_url):

    print("\nStarting download...\n")

    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": "downloads/video.%(ext)s",
        "noplaylist": True,
        "quiet": False,
        "geo_bypass": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url])

        print("\n✅ Download completed!")

    except Exception as e:
        print("\n❌ Download failed!")
        print(e)

In [8]:
download_video(youtube_url)


Starting download...

[youtube] Extracting URL: https://www.youtube.com/watch?v=eVFzbxmKNUw
[youtube] eVFzbxmKNUw: Downloading webpage


[youtube] eVFzbxmKNUw: Downloading android vr player API JSON
[info] eVFzbxmKNUw: Downloading 1 format(s): 401+251
[download] Destination: downloads/video.f401.mp4
[download] 100% of  221.34MiB in 00:00:06 at 31.62MiB/s  
[download] Destination: downloads/video.f251.webm
[download] 100% of   11.46MiB in 00:00:00 at 20.05MiB/s  
[Merger] Merging formats into "downloads/video.mp4"
Deleting original file downloads/video.f251.webm (pass -k to keep)
Deleting original file downloads/video.f401.mp4 (pass -k to keep)

✅ Download completed!


In [9]:
import os

video_path = None

for file in os.listdir("downloads"):
    if file.endswith((".mp4", ".mkv", ".webm", ".mov")):
        video_path = os.path.join("downloads", file)
        break

if video_path:
    print("✅ Video Found:")
    print(video_path)
else:
    print("❌ No video found!")

✅ Video Found:
downloads/video.mp4


In [10]:
audio_path = "audio/audio.wav"

command = [
    "ffmpeg",
    "-i",
    video_path,
    "-ar",
    "16000",
    "-ac",
    "1",
    audio_path,
    "-y"
]

subprocess.run(command)

print("✅ Audio Extracted Successfully!")
print(audio_path)

✅ Audio Extracted Successfully!
audio/audio.wav


In [11]:
import os

if os.path.exists(audio_path):
    print("✅ Audio file created successfully!")
else:
    print("❌ Audio extraction failed.")

✅ Audio file created successfully!


In [12]:
import whisper

print("Loading Whisper model...")

# Choose one:
# tiny   -> Fastest, least accurate
# base   -> Good for testing
# small  -> Better accuracy
# medium -> Even better (requires more GPU memory)

model = whisper.load_model("base")

print("✅ Whisper model loaded successfully!")

Loading Whisper model...


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 48.7MiB/s]


✅ Whisper model loaded successfully!


In [13]:
print("Transcribing audio...")

result = model.transcribe(
    audio_path,
    language="en"
)

segments = result["segments"]

print(f"✅ Transcription completed!")
print(f"Total segments: {len(segments)}")

Transcribing audio...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Transcription completed!
Total segments: 243


In [14]:

import pandas as pd

transcript = []

for seg in segments:
    transcript.append({
        "Start": seg["start"],
        "End": seg["end"],
        "Text": seg["text"].strip()
    })

df = pd.DataFrame(transcript)

df.head(10)

,Start,End,Text
0,0.00,8.48,"[♪ Music playing in background, applause and c..."
1,8.48,9.72,PICTURE THIS!
2,9.72,12.46,You're going on a boat trip.
3,12.46,16.38,And you get on board with your family and you ...
4,16.38,19.08,"And the captain comes out to greet you and says,"
5,19.08,24.32,"Hi, my name's Montana, a bonfeless."
6,24.32,29.30,"Oh, so I'll be your captain for this journey."
7,29.30,33.76,"Oh, boy, let's just have a great trip."
8,33.76,34.76,Sorry.
9,34.76,40.80,"Nope, get me off of this boat."


In [16]:
import pandas as pd

rows = []

for seg in segments:

    rows.append({

        "start": float(seg["start"]),
        "end": float(seg["end"]),
        "duration": round(seg["end"]-seg["start"],2),
        "text": seg["text"]

    })

df = pd.DataFrame(rows)

print("Total Segments :",len(df))

df.head()

Total Segments : 243


,start,end,duration,text
0,0.00,8.48,8.48,"[♪ Music playing in background, applause and ..."
1,8.48,9.72,1.24,PICTURE THIS!
2,9.72,12.46,2.74,You're going on a boat trip.
3,12.46,16.38,3.92,And you get on board with your family and you...
4,16.38,19.08,2.70,"And the captain comes out to greet you and says,"


In [17]:
import re

def clean_text(text):

    text = str(text)

    text = text.replace("\n"," ")

    text = re.sub(r"\s+"," ",text)

    text = text.strip()

    return text

df["clean_text"] = df["text"].apply(clean_text)

df.head()

,start,end,duration,text,clean_text
0,0.00,8.48,8.48,"[♪ Music playing in background, applause and ...","[♪ Music playing in background, applause and c..."
1,8.48,9.72,1.24,PICTURE THIS!,PICTURE THIS!
2,9.72,12.46,2.74,You're going on a boat trip.,You're going on a boat trip.
3,12.46,16.38,3.92,And you get on board with your family and you...,And you get on board with your family and you ...
4,16.38,19.08,2.70,"And the captain comes out to greet you and says,","And the captain comes out to greet you and says,"


In [18]:

before = len(df)

df = df[df["clean_text"]!=""]

after = len(df)

print("Removed :",before-after)
print("Remaining :",after)

Removed : 0
Remaining : 243


In [19]:
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

df = df[df["word_count"]>=3]

print("Remaining Segments :",len(df))

Remaining Segments : 222


In [20]:
before = len(df)

df = df.drop_duplicates(subset="clean_text")

after = len(df)

print("Duplicates Removed :",before-after)

Duplicates Removed : 0


In [21]:
df = df.reset_index(drop=True)

df.head()

,start,end,duration,text,clean_text,word_count
0,0.00,8.48,8.48,"[♪ Music playing in background, applause and ...","[♪ Music playing in background, applause and c...",14
1,9.72,12.46,2.74,You're going on a boat trip.,You're going on a boat trip.,6
2,12.46,16.38,3.92,And you get on board with your family and you...,And you get on board with your family and you ...,13
3,16.38,19.08,2.70,"And the captain comes out to greet you and says,","And the captain comes out to greet you and says,",10
4,19.08,24.32,5.24,"Hi, my name's Montana, a bonfeless.","Hi, my name's Montana, a bonfeless.",6


In [22]:
merged = []

i = 0

while i < len(df):

    current = df.iloc[i].copy()

    while (

        i+1 < len(df)

        and current["word_count"]<8

    ):

        nxt = df.iloc[i+1]

        current["clean_text"] += " " + nxt["clean_text"]

        current["end"] = nxt["end"]

        current["duration"] = round(

            current["end"]-current["start"],2

        )

        current["word_count"] = len(

            current["clean_text"].split()

        )

        i += 1

    merged.append(current)

    i += 1

merged_df = pd.DataFrame(merged)

print("Merged Segments :",len(merged_df))

Merged Segments : 153


In [23]:
def timestamp(sec):

    m = int(sec//60)

    s = int(sec%60)

    return f"{m:02d}:{s:02d}"

merged_df["timestamp"] = merged_df["start"].apply(timestamp)

merged_df.head()

,start,end,duration,text,clean_text,word_count,timestamp
0,0.00,8.48,8.48,"[♪ Music playing in background, applause and ...","[♪ Music playing in background, applause and c...",14,00:00
1,9.72,16.38,6.66,You're going on a boat trip.,You're going on a boat trip. And you get on bo...,19,00:09
3,16.38,19.08,2.70,"And the captain comes out to greet you and says,","And the captain comes out to greet you and says,",10,00:16
4,19.08,29.30,10.22,"Hi, my name's Montana, a bonfeless.","Hi, my name's Montana, a bonfeless. Oh, so I'l...",15,00:19
6,29.30,33.76,4.46,"Oh, boy, let's just have a great trip.","Oh, boy, let's just have a great trip.",8,00:29


In [24]:
merged_df.to_csv(

    "transcripts/clean_transcript.csv",

    index=False

)

print("Saved Successfully")

Saved Successfully


In [25]:
merged_df[
    [
        "timestamp",
        "clean_text",
        "word_count"
    ]
].head(20)

,timestamp,clean_text,word_count
0,00:00,"[♪ Music playing in background, applause and c...",14
1,00:09,You're going on a boat trip. And you get on bo...,19
3,00:16,"And the captain comes out to greet you and says,",10
4,00:19,"Hi, my name's Montana, a bonfeless. Oh, so I'l...",15
6,00:29,"Oh, boy, let's just have a great trip.",8
7,00:34,"Nope, get me off of this boat. What we want in...",27
9,00:47,I'll be your captain for this journey. Let's h...,12
11,00:52,"The point is, when you are the speaker, you ar...",18
12,00:57,"show up really matters. For the last 17 years,...",21
14,01:06,companies to small startups and everyone from ...,15


In [26]:
conversation_data = []

for i in range(len(merged_df)):

    previous_text = ""
    current_text = merged_df.iloc[i]["clean_text"]
    next_text = ""

    if i > 0:
        previous_text = merged_df.iloc[i-1]["clean_text"]

    if i < len(merged_df)-1:
        next_text = merged_df.iloc[i+1]["clean_text"]

    conversation = f"""
Previous:
{previous_text}

Current:
{current_text}

Next:
{next_text}
"""

    conversation_data.append({
        "start": merged_df.iloc[i]["start"],
        "end": merged_df.iloc[i]["end"],
        "timestamp": merged_df.iloc[i]["timestamp"],
        "conversation": conversation
    })

conversation_df = pd.DataFrame(conversation_data)

print("Conversation windows created:", len(conversation_df))

Conversation windows created: 153


In [27]:
conversation_df.head(5)

,start,end,timestamp,conversation
0,0.00,8.48,00:00,\nPrevious:\n\n\nCurrent:\n[♪ Music playing in...
1,9.72,16.38,00:09,"\nPrevious:\n[♪ Music playing in background, a..."
2,16.38,19.08,00:16,\nPrevious:\nYou're going on a boat trip. And ...
3,19.08,29.30,00:19,\nPrevious:\nAnd the captain comes out to gree...
4,29.30,33.76,00:29,"\nPrevious:\nHi, my name's Montana, a bonfeles..."


In [28]:
print(conversation_df.loc[5, "conversation"])


Previous:
Oh, boy, let's just have a great trip.

Current:
Nope, get me off of this boat. What we want in that moment is for the captain to walk out and say, Hi, my name is Montana

Next:
I'll be your captain for this journey. Let's have a great trip.



In [29]:
def count_words(text):
    return len(text.split())

def question_mark(text):
    return int("?" in text)

conversation_df["word_count"] = merged_df["clean_text"].apply(count_words)

conversation_df["contains_question"] = merged_df["clean_text"].apply(question_mark)

conversation_df["duration"] = merged_df["duration"]

conversation_df.head()

,start,end,timestamp,conversation,word_count,contains_question,duration
0,0.00,8.48,00:00,\nPrevious:\n\n\nCurrent:\n[♪ Music playing in...,14.0,0.0,8.48
1,9.72,16.38,00:09,"\nPrevious:\n[♪ Music playing in background, a...",19.0,0.0,6.66
2,16.38,19.08,00:16,\nPrevious:\nYou're going on a boat trip. And ...,NaN,NaN,NaN
3,19.08,29.30,00:19,\nPrevious:\nAnd the captain comes out to gree...,10.0,0.0,2.70
4,29.30,33.76,00:29,"\nPrevious:\nHi, my name's Montana, a bonfeles...",15.0,0.0,10.22


In [30]:
agreement_words = [
    "yes",
    "yeah",
    "exactly",
    "correct",
    "right",
    "absolutely",
    "true",
    "definitely",
    "agree"
]

def agreement_score(text):

    text = text.lower()

    score = 0

    for word in agreement_words:
        if word in text:
            score += 1

    return score

conversation_df["agreement_score"] = merged_df["clean_text"].apply(agreement_score)

In [34]:
disagreement_words = [
    "no",
    "not",
    "never",
    "don't",
    "disagree",
    "wrong",
    "however",
    "actually",
    "but"
]

def disagreement_score(text):

    text = text.lower()

    score = 0

    for word in disagreement_words:
        if word in text:
            score += 1

    return score

conversation_df["disagreement_score"] = merged_df["clean_text"].apply(disagreement_score)

In [32]:
question_words = [
    "what",
    "why",
    "when",
    "where",
    "who",
    "which",
    "how",
    "can",
    "could",
    "should",
    "would",
    "do",
    "does",
    "did"
]

def question_score(text):

    text = text.lower()

    score = 0

    for word in question_words:
        if word in text:
            score += 1

    if "?" in text:
        score += 2

    return score

conversation_df["question_score"] = merged_df["clean_text"].apply(question_score)

In [35]:
conversation_df[
    [
        "timestamp",
        "word_count",
        "contains_question",
        "question_score",
        "agreement_score",
        "disagreement_score"
    ]
].head(20)

,timestamp,word_count,contains_question,question_score,agreement_score,disagreement_score
0,00:00,14.0,0.0,0.0,0.0,0.0
1,00:09,19.0,0.0,0.0,0.0,0.0
2,00:16,NaN,NaN,NaN,NaN,NaN
3,00:19,10.0,0.0,0.0,0.0,0.0
4,00:29,15.0,0.0,0.0,0.0,0.0
5,00:34,NaN,NaN,NaN,NaN,NaN
6,00:47,8.0,0.0,0.0,0.0,0.0
7,00:52,27.0,0.0,1.0,0.0,1.0
8,00:57,NaN,NaN,NaN,NaN,NaN
9,01:06,12.0,0.0,0.0,0.0,0.0


In [36]:
conversation_df.to_csv(
    "transcripts/conversation_features.csv",
    index=False
)

print("✅ Feature dataset saved!")

✅ Feature dataset saved!


In [40]:
!pip -q install scikit-learn

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("TF-IDF Ready")

TF-IDF Ready


In [42]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30,
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(
    conversation_df["conversation"]
)

feature_names = vectorizer.get_feature_names_out()

print("Top Features Extracted:", len(feature_names))

Top Features Extracted: 30


In [43]:
print("Top Keywords:\n")

for word in feature_names:
    print(word)

Top Keywords:

audience
better
bring
captain
confident
current
day
don
feel
just
know
let
like
loud
make
new
pause
practice
previous
purpose
really
say
sentence
shape
silent
silent sentence
stage
superhero
time
want


In [44]:

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()

conversation_df["tfidf_score"] = tfidf_scores

conversation_df[
    [
        "timestamp",
        "tfidf_score"
    ]
].head()

,timestamp,tfidf_score
0,00:00,1.414214
1,00:09,1.528082
2,00:16,1.314115
3,00:19,1.839441
4,00:29,2.301157


In [45]:

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()

conversation_df["tfidf_score"] = tfidf_scores

conversation_df[
    [
        "timestamp",
        "tfidf_score"
    ]
].head()

,timestamp,tfidf_score
0,00:00,1.414214
1,00:09,1.528082
2,00:16,1.314115
3,00:19,1.839441
4,00:29,2.301157


In [47]:
conversation_df[
    [
        "timestamp",
        "tfidf_score",
        "conversation"
    ]
].head(10)

,timestamp,tfidf_score,conversation
0,00:00,1.414214,\nPrevious:\n\n\nCurrent:\n[♪ Music playing in...
1,00:09,1.528082,"\nPrevious:\n[♪ Music playing in background, a..."
2,00:16,1.314115,\nPrevious:\nYou're going on a boat trip. And ...
3,00:19,1.839441,\nPrevious:\nAnd the captain comes out to gree...
4,00:29,2.301157,"\nPrevious:\nHi, my name's Montana, a bonfeles..."
5,00:34,2.229624,"\nPrevious:\nOh, boy, let's just have a great ..."
6,00:47,1.929103,"\nPrevious:\nNope, get me off of this boat. Wh..."
7,00:52,1.900395,\nPrevious:\nI'll be your captain for this jou...
8,00:57,2.042373,"\nPrevious:\nThe point is, when you are the sp..."
9,01:06,2.481118,\nPrevious:\nshow up really matters. For the l...


In [48]:
!pip -q install vaderSentiment

In [49]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return abs(analyzer.polarity_scores(text)["compound"])

conversation_df["sentiment_score"] = conversation_df["conversation"].apply(get_sentiment)

conversation_df[["timestamp","sentiment_score"]].head()

,timestamp,sentiment_score
0,00:00,0.7845
1,00:09,0.8481
2,00:16,0.3182
3,00:19,0.7506
4,00:29,0.6597


In [51]:
features = [

    "question_score",
    "agreement_score",
    "disagreement_score",
    "word_count",
    "duration",
    "sentiment_score",
    "tfidf_score"

]

for feature in features:

    max_val = conversation_df[feature].max()

    if max_val > 0:

        conversation_df[feature] = conversation_df[feature] / max_val

print("Normalization Complete")

Normalization Complete


In [52]:

WEIGHTS = {

    "question" : 0.25,
    "agreement" : 0.15,
    "disagreement" : 0.15,
    "tfidf" : 0.20,
    "sentiment" : 0.15,
    "word" : 0.05,
    "duration" : 0.05

}

In [53]:
conversation_df["highlight_score"] = (

WEIGHTS["question"] * conversation_df["question_score"]

+

WEIGHTS["agreement"] * conversation_df["agreement_score"]

+

WEIGHTS["disagreement"] * conversation_df["disagreement_score"]

+

WEIGHTS["tfidf"] * conversation_df["tfidf_score"]

+

WEIGHTS["sentiment"] * conversation_df["sentiment_score"]

+

WEIGHTS["word"] * conversation_df["word_count"]

+

WEIGHTS["duration"] * conversation_df["duration"]

)

In [54]:
conversation_df["highlight_score"] = (

conversation_df["highlight_score"] * 100

).round(2)

In [55]:
def classify(row):

    if row["question_score"] >= 0.40:
        return "Question"

    elif row["agreement_score"] >= row["disagreement_score"] and row["agreement_score"] >= 0.30:
        return "Agreement"

    elif row["disagreement_score"] >= 0.30:
        return "Disagreement"

    else:
        return "Other"

conversation_df["event"] = conversation_df.apply(classify, axis=1)

In [56]:

ranked_df = conversation_df.sort_values(

    by="highlight_score",

    ascending=False

).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

ranked_df.head(20)


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,question_score,disagreement_score,sentiment_score,tfidf_score,highlight_score,event,rank
0,579.56,582.64,09:39,\nPrevious:\nWhat would be a better message? A...,0.740741,0.0,0.584718,0.0,1.000000,0.333333,1.000000,0.777406,67.18,Question,1
1,571.72,579.56,09:31,\nPrevious:\nAnd that really was not working. ...,0.592593,0.0,0.674419,0.0,1.000000,0.333333,0.950584,0.714614,64.89,Question,2
2,751.44,759.92,12:31,\nPrevious:\nOr maybe you're just super excite...,0.555556,0.0,0.734219,1.0,0.285714,0.666667,0.940763,0.589742,64.50,Agreement,3
3,595.68,600.24,09:55,\nPrevious:\nWhat is my deeper purpose here? A...,0.592593,0.0,0.498339,0.0,0.619048,0.666667,0.944485,0.821019,61.52,Question,4
4,848.44,854.16,14:08,\nPrevious:\nWhat's being broadcast? Then repl...,0.370370,1.0,0.308970,0.0,0.428571,1.000000,0.904373,0.864589,59.97,Question,5
5,568.00,571.72,09:28,\nPrevious:\nI realized I was giving myself an...,0.703704,0.0,0.581395,0.0,0.809524,0.333333,0.933526,0.654893,58.76,Question,6
6,199.92,203.40,03:19,"\nPrevious:\nNumber three, superhero stance. I...",0.481481,0.0,0.504983,1.0,0.714286,0.333333,0.474310,0.413448,58.17,Question,7
7,207.84,212.00,03:27,"\nPrevious:\nYes, let that change your posture...",0.666667,0.0,0.362126,1.0,0.095238,0.333333,0.971053,0.711291,56.32,Agreement,8
8,449.20,453.08,07:29,\nPrevious:\nread your music every day and the...,0.703704,0.0,0.807309,0.0,0.428571,0.666667,0.941797,0.666401,55.72,Question,9
9,252.36,255.16,04:12,"\nPrevious:\nAnd if you need an easy shortcut,...",0.518519,1.0,0.508306,1.0,0.095238,0.333333,0.790861,0.760900,54.60,Agreement,10


In [60]:
import os

os.makedirs("output", exist_ok=True)

ranked_df.to_csv(

    "WEEK1_HIGHLIGHT EXTRACTION.csv",

    index=False

)

print("✅ Saved Successfully!")

✅ Saved Successfully!


In [58]:
ranked_df[

[
    "rank",
    "timestamp",
    "event",
    "highlight_score",
    "conversation"

]

].head(20)

,rank,timestamp,event,highlight_score,conversation
0,1,09:39,Question,67.18,\nPrevious:\nWhat would be a better message? A...
1,2,09:31,Question,64.89,\nPrevious:\nAnd that really was not working. ...
2,3,12:31,Agreement,64.50,\nPrevious:\nOr maybe you're just super excite...
3,4,09:55,Question,61.52,\nPrevious:\nWhat is my deeper purpose here? A...
4,5,14:08,Question,59.97,\nPrevious:\nWhat's being broadcast? Then repl...
5,6,09:28,Question,58.76,\nPrevious:\nI realized I was giving myself an...
6,7,03:19,Question,58.17,"\nPrevious:\nNumber three, superhero stance. I..."
7,8,03:27,Agreement,56.32,"\nPrevious:\nYes, let that change your posture..."
8,9,07:29,Question,55.72,\nPrevious:\nread your music every day and the...
9,10,04:12,Agreement,54.60,"\nPrevious:\nAnd if you need an easy shortcut,..."


In [61]:
from google.colab import files

files.download("WEEK1_HIGHLIGHT EXTRACTION.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>